# 🥇 Gold Layer Pipeline (Star Schema Generation)
**الهدف:** تحويل البيانات الموحدة والمحسوبة من طبقة السيلفر (`silver_layer.csv`) إلى تصميم النجمة (**Star Schema**)، عن طريق فصلها إلى جداول أبعاد (**Dimensions**) وجدول حقائق (**Fact**) جاهز ومثالي للرفع والتحليل المباشر في **Power BI**.

### 🌟 هيكل البيانات (Star Schema):
1. **`dim_location`**: يحتوي على الأماكن الفريدة بشكل فريد ومفتاح تعريفي (`location_key`) لكل مكان.
2. **`dim_date`**: يحتوي على التواريخ الفريدة والأبعاد الزمنية الذكية (السنة، الشهر، الفصل، أيام العمل) لتسهيل الـ Hierarchy.
3. **`fact_airquality_weather`**: يحتوي على المقاييس الرقمية الموحدة ومؤشرات الـ AQI والطقس مرتبطة بالمفاتيح الرقمية فقط للتوفير في المساحة وزيادة الأداء.

## ⚙️ 1. إعداد البيئة وتحميل البيانات

In [ ]:
import pandas as pd
import numpy as np
import os

# تحديد مسارات الملفات والمجلدات
SILVER_FILE = 'silver_layer.csv'
GOLD_DIR = 'gold_layer/'

if not os.path.exists(GOLD_DIR):
    os.makedirs(GOLD_DIR)

print(f'[GOLD] Loading cleaned Silver data from: {SILVER_FILE}')
df = pd.read_csv(SILVER_FILE)
df['date'] = pd.to_datetime(df['date'])
print(f'[GOLD] ✓ Loaded {len(df):,} rows successfully.')

## 📍 2. بناء جدول أبعاد المكان (`Dim_Location`)

In [ ]:
print('[GOLD] Extracting unique locations...')
dim_location = df[['country', 'city']].drop_duplicates().reset_index(drop=True)

# إنشاء المفتاح السري التعريفي (Surrogate Key)
dim_location['location_key'] = dim_location.index + 1

# إعادة ترتيب الأعمدة بشكل احترافي
dim_location = dim_location[['location_key', 'country', 'city']]

print(f'[GOLD] ✓ Dim_Location built with {len(dim_location)} unique locations.')
dim_location.head()

## 📅 3. بناء جدول أبعاد الزمان (`Dim_Date`)

In [ ]:
print('[GOLD] Extracting time intelligence dimensions...')
date_cols = ['date', 'is_weekday', 'month_name', 'season']
dim_date = df[date_cols].drop_duplicates().reset_index(drop=True)

# تفكيك التاريخ إلى أبعاد إضافية لتسهيل عمل الـ Hierarchies في الـ Power BI
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['day'] = dim_date['date'].dt.day

# إعادة الترتيب الهيكلي للجدول
dim_date = dim_date[['date', 'year', 'month', 'month_name', 'day', 'is_weekday', 'season']]

print(f'[GOLD] ✓ Dim_Date built with {len(dim_date)} unique dates.')
dim_date.head()

## 📊 4. بناء جدول الحقائق والمقاييس (`Fact_AirQuality_Weather`)

In [ ]:
print('[GOLD] Mapping location keys into Fact Table...')
# ربط جدول الفاكت بجدول المكان للحصول على الـ Key المقابل لكل مدينة
fact_df = pd.merge(df, dim_location, on=['country', 'city'], how='left')

# تحديد المقاييس الرقمية المتاحة للطقس
weather_measures = ['temperature', 'pressure', 'humidity', 'dew', 'precipitation', 'wind gust', 'uvi']
actual_weather = [c for c in weather_measures if c in fact_df.columns]
aqi_measures = ['AQI', 'AQI_label', 'AQI_sort_order']

# اختيار الأعمدة النهائية لجدول الـ Fact (إقصاء النصوص والأسماء لتوفير المساحة تماشياً مع الـ Data Modeling)
fact_columns = ['date', 'location_key'] + actual_weather + aqi_measures
fact_table = fact_df[fact_columns].copy()

# إضافة Surrogate Key خاص بجدول الحقائق كـ Best Practice
fact_table.insert(0, 'fact_key', fact_table.index + 1)

print(f'[GOLD] ✓ Fact Table built with {len(fact_table)} rows.')
fact_table.head()

## 💾 5. تصدير ملفات الـ Star Schema لطبقة الـ Gold

In [ ]:
print('[GOLD] Exporting Star Schema tables to CSV format...')

dim_location.to_csv(os.path.join(GOLD_DIR, 'dim_location.csv'), index=False, encoding='utf-8')
dim_date.to_csv(os.path.join(GOLD_DIR, 'dim_date.csv'), index=False, encoding='utf-8')
fact_table.to_csv(os.path.join(GOLD_DIR, 'fact_airquality_weather.csv'), index=False, encoding='utf-8')

print('\n' + '='*60)
print('  ✓ GOLD LAYER GENERATED SUCCESSFULLY!')
print(f'  All files saved in directory: "{GOLD_DIR}"')
print('='*60)